# 第 16 章 · SARSA 与 Q-learning：用 TD 学 q，学会怎么做

本章回答：TD 只学会了"评估"，怎么用它来找最好的动作？SARSA 和 Q-learning 只差一个字，结果为什么会走出两条完全不同的路线？on-policy 和 off-policy 是什么意思？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/16-sarsa-qlearning.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 手算一步 | `r + Q[ns][na]` vs `r + Q[ns].max()` | −6.0 vs −5.0 |
| 悬崖，30 次 × 500 局 | `run("sarsa")` / `run("q")` | 最后 100 局平均 −27.2 / −51.3 |
| 贪心路线经过哪几行 | `greedy_path(Q)` | SARSA [0, 1, 2]，Q-learning [2] |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `CLIFF_ROWS`, `CLIFF_COLS`**（来自 `rl_envs.py`）

In [ ]:
CLIFF_ROWS, CLIFF_COLS = 4, 12

**常量 `CLIFF_START`, `CLIFF_GOAL`**（来自 `rl_envs.py`）

In [ ]:
CLIFF_START, CLIFF_GOAL = (3, 0), (3, 11)

**常量 `MOVE_NAMES`**（来自 `rl_envs.py`）

In [ ]:
MOVE_NAMES = ["↑", "↓", "→", "←"]

**常量 `MOVES`**（来自 `rl_envs.py`）

In [ ]:
MOVES = [(-1, 0), (1, 0), (0, 1), (0, -1)]

**函数 `cliff_step`**（来自 `rl_envs.py`）：每步 −1；掉下悬崖（最下一行中间 10 格）−100 并回到起点；到达终点结束。

In [ ]:
def cliff_step(state, a):
    """每步 −1；掉下悬崖（最下一行中间 10 格）−100 并回到起点；到达终点结束。"""
    r, c = state
    dr, dc = MOVES[a]
    nr, nc = min(max(r + dr, 0), CLIFF_ROWS - 1), min(max(c + dc, 0), CLIFF_COLS - 1)
    if nr == 3 and 1 <= nc <= 10:
        return CLIFF_START, -100.0, False
    return (nr, nc), -1.0, (nr, nc) == CLIFF_GOAL

**函数 `greedy_path`**（来自 `ch16_sarsa_qlearning.py`）：下面要用到的小工具。

In [ ]:
def greedy_path(Q):
    s, path = CLIFF_START, [CLIFF_START]
    for _ in range(60):
        s, _, done = cliff_step(s, int(np.argmax(Q[s])))
        path.append(s)
        if done:
            break
    return path

**函数 `eps_greedy`**（来自 `ch16_sarsa_qlearning.py`）：下面要用到的小工具。

In [ ]:
def eps_greedy(Q, s, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(4))
    q = Q[s]
    return int(rng.choice(np.flatnonzero(q == q.max())))

**函数 `run`**（来自 `ch16_sarsa_qlearning.py`）：下面要用到的小工具。

In [ ]:
def run(method, episodes=500, alpha=0.5, eps=0.1, gamma=1.0, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((CLIFF_ROWS, CLIFF_COLS, 4))
    sums = []
    for _ in range(episodes):
        s, total = CLIFF_START, 0.0
        a = eps_greedy(Q, s, eps, rng)
        for _ in range(10000):
            ns, r, done = cliff_step(s, a)
            total += r
            na = eps_greedy(Q, ns, eps, rng)
            if method == "sarsa":
                target = r + (0 if done else gamma * Q[ns][na])  # 用真正要做的下一个动作
            else:
                target = r + (0 if done else gamma * Q[ns].max())  # 用下一格最好的动作
            Q[s][a] += alpha * (target - Q[s][a])
            s, a = ns, na
            if done:
                break
        sums.append(total)
    return Q, np.array(sums)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
Q = np.zeros((CLIFF_ROWS, CLIFF_COLS, 4))

In [ ]:
Q[2, 0] = [-5, -12, -4, -6]

In [ ]:
s, a, r, ns, na = (3, 0), 0, -1.0, (2, 0), 0

In [ ]:
sarsa_t = r + Q[ns][na]

In [ ]:
q_t = r + Q[ns].max()

In [ ]:
print(f"手算一步：SARSA 目标 = {sarsa_t}（下一步真做 ↑），Q-learning 目标 = {q_t}（下一格最好 →）")

In [ ]:
for m in ["sarsa", "q"]:
        res = [run(m, seed=k) for k in range(30)]
        tail = np.mean([r[1][-100:].mean() for r in res])
        rows = {p[0] for p in greedy_path(res[0][0])[1:-1]}
        print(f"{m}: 最后 100 局平均每局奖励 {tail:.1f}；贪心路线经过的行 {sorted(rows)}")

In [ ]:
print("动作编号：", MOVE_NAMES, "终点", CLIFF_GOAL)

## 3. 代码里才看得到的

`eps_greedy` 在并列时用 `rng.choice(np.flatnonzero(q == q.max()))` 随机挑（第 13 章附录提到的问题）。一开始 Q 全是 0，如果总挑编号最小的"↑"，智能体会在起点一直往上撞墙，学得很慢。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 换步长（30 次实验，最后 100 局平均）：α = 0.1 → SARSA −21.8、Q-learning −47.5；α = 0.9 → SARSA −72.3、Q-learning −51.1。α 太大时 SARSA 反而更差。
2. 用 `greedy_path(Q)` 打出两种方法学到的路线，自己在纸上画出来，和讲义的图对比。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 不知道 p 学 Q 不学 V，挑动作看 Q 最大；
> SARSA 用真要做的下一步，学的就是正在做的（同策略）；
> Q-learning 用下一格最好的，学最优、做 ε-greedy（异策略）；
> 悬崖边：SARSA 知道自己手抖绕远路，Q-learning 贴边常掉崖。